In [1]:
%pip install --quiet qiskit qiskit-aer qiskit-ibm-runtime numpy scipy matplotlib pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scipy.optimize
from collections import deque
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
import os

print("Imports successful!")

Imports successful!


In [3]:
IBM_QUANTUM_TOKEN = "bpY9c7MbWpOlcCihOXivSb_4BRkLBMPLBitn85qpzLWZ"
QiskitRuntimeService.save_account(channel="ibm_quantum_platform",
                                  token=IBM_QUANTUM_TOKEN, # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
                                  instance="crn:v1:bluemix:public:quantum-computing:us-east:a/c599629df45c4794a55daad4835d0856:b209ad37-f8fb-4fc5-9541-0ffc4f4fb7bf::",
                                  name="open",
                                  overwrite=True)

In [6]:
# Qiskit Runtime migration: use new service initialization (no channel argument), and recommend token via env or save_account
from qiskit_ibm_runtime import QiskitRuntimeService
import os

# Option 1: Set token as environment variable (recommended, do this before running the notebook)
# os.environ["QISKIT_IBM_TOKEN"] = "YOUR_TOKEN"

# Option 2: Save account (only once per environment, then comment out)
# QiskitRuntimeService.save_account(token="YOUR_TOKEN", overwrite=True)

# Initialize the service with your instance string
service = QiskitRuntimeService(instance="crn:v1:bluemix:public:quantum-computing:us-east:a/c599629df45c4794a55daad4835d0856:b209ad37-f8fb-4fc5-9541-0ffc4f4fb7bf::")

# Backend: change here if needed
backend = service.backend("ibm_fez")
print(f"Connected to backend: {backend.name}")
print(f"Basis gates: {backend.basis_gates}")


Connected to backend: ibm_fez
Basis gates: ['cz', 'id', 'rz', 'sx', 'x']


In [9]:
job_id = "d5nnjhjh36vs73bjht9g"
retrieved_job = service.job(job_id)
retrieved_job.result()
result = retrieved_job.result()

In [10]:
# Print the structure of the result object and extract counts from BitArray
print("Type of result:", type(result))
if hasattr(result, '__getitem__') and len(result) > 0:
    print("Type of result[0]:", type(result[0]))
    if hasattr(result[0], 'data') and hasattr(result[0].data, 'c'):
        print("Type of result[0].data.c:", type(result[0].data.c))
        bitarray = result[0].data.c
        # Try to convert BitArray to numpy array using .to_numpy() or .tolist()
        arr = None
        if hasattr(bitarray, 'to_numpy'):
            arr = bitarray.to_numpy()
        elif hasattr(bitarray, 'tolist'):
            arr = np.array(bitarray.tolist())
        else:
            print("BitArray has no to_numpy() or tolist() method.")
        if arr is not None:
            # arr shape: (num_shots, num_bits) or (num_shots,) for 1 bit
            if arr.ndim == 1:
                # Single bit per shot
                bitstrings = [str(b) for b in arr]
            else:
                # Multi-bit: join bits per shot
                bitstrings = ["".join(str(b) for b in row) for row in arr]
            from collections import Counter
            counts = dict(Counter(bitstrings))
            print("Extracted counts:", counts)
            total_shots = sum(counts.values())
            prob_0 = counts.get('0', 0) / total_shots if total_shots > 0 else 0.0
            print(f"Probability of '0': {prob_0:.6f}")
        else:
            print("Could not convert BitArray to numpy array.")
    else:
        print("result[0].data.c not found.")
else:
    print("Result object is empty or not subscriptable.")

Type of result: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>
Type of result[0]: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>
Type of result[0].data.c: <class 'qiskit.primitives.containers.bit_array.BitArray'>
BitArray has no to_numpy() or tolist() method.
Could not convert BitArray to numpy array.


In [11]:
# --- Diagnostic: Explore BitArray attributes and try extraction ---
if hasattr(result, '__getitem__') and len(result) > 0 and hasattr(result[0], 'data') and hasattr(result[0].data, 'c'):
    bitarray = result[0].data.c
    print("Type of bitarray:", type(bitarray))
    print("dir(bitarray):", dir(bitarray))
    # Try common attributes
    for attr in ['data', '_data', '__array__', 'array', 'buffer', 'tolist', 'to_numpy', 'asarray']:
        if hasattr(bitarray, attr):
            print(f"bitarray.{attr}: ", getattr(bitarray, attr))
    # Try to iterate or index
    try:
        print("First 5 elements (bitarray[:5]):", bitarray[:5])
    except Exception as e:
        print("Indexing failed:", e)
    try:
        print("List conversion: ", list(bitarray)[:5])
    except Exception as e:
        print("List conversion failed:", e)
    # Try numpy.array()
    try:
        arr = np.array(bitarray)
        print("np.array(bitarray) shape:", arr.shape)
        print("First 5 rows:", arr[:5])
    except Exception as e:
        print("np.array conversion failed:", e)
else:
    print("BitArray not found in result object.")

Type of bitarray: <class 'qiskit.primitives.containers.bit_array.BitArray'>
dir(bitarray): ['__abstractmethods__', '__and__', '__annotations__', '__callable_proto_members_only__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__invert__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__or__', '__parameters__', '__protocol_attrs__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '__xor__', '_abc_impl', '_array', '_bytes_to_bitstring', '_bytes_to_int', '_get_counts', '_is_protocol', '_is_runtime_protocol', '_num_bits', '_prepare_broadcastable', '_shape', 'array', 'bitcount', 'concatenate', 'concatenate_bits', 'concatenate_shots', 'expectation_values', 'from_bool_array', 'from_counts', 'from_samples', 'get_bitstrings', 'g

In [12]:
# Extract measurement counts from BitArray using .array attribute
if hasattr(result, '__getitem__') and len(result) > 0 and hasattr(result[0], 'data') and hasattr(result[0].data, 'c'):
    bitarray = result[0].data.c
    arr = bitarray.array  # This is a numpy array of shape (num_shots, num_bits)
    print("Shape of bitarray.array:", arr.shape)
    if arr.ndim == 1 or arr.shape[1] == 1:
        # Single bit per shot
        bitstrings = [str(b[0]) if arr.ndim > 1 else str(b) for b in arr]
    else:
        # Multi-bit: join bits per shot
        bitstrings = ["".join(str(b) for b in row) for row in arr]
    from collections import Counter
    counts = dict(Counter(bitstrings))
    print("Extracted counts:", counts)
    total_shots = sum(counts.values())
    prob_0 = counts.get('0', 0) / total_shots if total_shots > 0 else 0.0
    print(f"Probability of '0': {prob_0:.6f}")
else:
    print("BitArray not found in result object.")

Shape of bitarray.array: (100, 1)
Extracted counts: {'0': 100}
Probability of '0': 1.000000


In [20]:
# --- Save all job IDs from a list of job objects to a text file ---
# jobs should be your list of job objects (e.g., from service.jobs() or similar)
with open('job_ids.txt', 'w') as f:  # 'w' to overwrite, 'a' to append
    for job in jobs_in:
        job_id = job.job_id() if hasattr(job, 'job_id') else job.job_id
        f.write(job_id + '\n')
print(f'Saved {len(jobs_in)} job IDs to job_ids.txt')

Saved 10 job IDs to job_ids.txt


In [33]:
import pandas as pd

csv_path = '/workspaces/qc_202526/all_time-workloads.csv'
df = pd.read_csv(csv_path)
print("CSV columns:", df.columns.tolist())

#Saving the ids in an array
if 'WorkloadId' in df.columns:
    job_ids = df['WorkloadId'].astype(str).tolist()
    print(f"Loaded {len(job_ids)} job IDs from CSV.")

CSV columns: ['WorkloadId', 'Status', 'Instance', 'Region', 'Mode', 'Created', 'Completed', 'QPU', 'Usage (seconds)', 'User', 'Tags', 'Account']
Loaded 45 job IDs from CSV.


save the job results for every id

In [31]:
results = []

for i in range(len(job_ids)):
    job_id = job_ids[i]
    retrieved_job = service.job(job_id)
    result = retrieved_job.result()
    results.append(result)


In [ ]:
# --- Extract measurement counts from all results in the results list ---
all_counts = []
for i, result in enumerate(results):
    try:
        if hasattr(result, '__getitem__') and len(result) > 0 and hasattr(result[0], 'data') and hasattr(result[0].data, 'c'):
            bitarray = result[0].data.c
            arr = bitarray.array  # numpy array (num_shots, num_bits)
            if arr.ndim == 1 or arr.shape[1] == 1:
                bitstrings = [str(b[0]) if arr.ndim > 1 else str(b) for b in arr]
            else:
                bitstrings = ["".join(str(b) for b in row) for row in arr]
            from collections import Counter
            counts = dict(Counter(bitstrings))
            all_counts.append(counts)
            print(f"Result {i}: counts = {counts}")
        else:
            print(f"Result {i}: BitArray not found in result object.")
            all_counts.append(None)
    except Exception as e:
        print(f"Result {i}: Error extracting counts: {e}")
        all_counts.append(None)
print(f"Extracted counts for {len(all_counts)} results.")

Result 0: counts = {'0': 100}
Result 1: counts = {'0': 100}
Result 2: counts = {'0': 100}
Result 3: counts = {'0': 100}
Result 4: counts = {'0': 100}
Result 5: counts = {'0': 100}
Result 6: counts = {'0': 100}
Result 7: counts = {'0': 100}
Result 8: counts = {'0': 100}
Result 9: counts = {'0': 100}
Result 10: counts = {'0': 100}
Result 11: counts = {'0': 100}
Result 12: counts = {'0': 100}
Result 13: counts = {'0': 100}
Result 14: counts = {'0': 100}
Result 15: counts = {'0': 100}
Result 16: counts = {'0': 100}
Result 17: counts = {'0': 99, '1': 1}
Result 18: counts = {'0': 100}
Result 19: counts = {'0': 100}
Result 20: counts = {'0': 100}
Result 21: counts = {'0': 100}
Result 22: counts = {'0': 100}
Result 23: counts = {'0': 99, '1': 1}
Result 24: counts = {'0': 100}
Result 25: counts = {'0': 100}
Result 26: counts = {'0': 99, '1': 1}
Result 27: counts = {'0': 100}
Result 28: counts = {'0': 100}
Result 29: counts = {'0': 100}
Result 30: counts = {'0': 100}
Result 31: counts = {'0': 10